In [1]:
import os
import cv2
import numpy as np
from PIL import Image, ImageChops, ImageEnhance, UnidentifiedImageError
from tqdm import tqdm
import tempfile

# ==============================
# CONFIGURATION
# ==============================
# IMPORTANT: Make sure this path points to your dataset folder
DATA_DIR = r"C:\Users\LENOVO\Desktop\main project\CASIA2" 
OUTPUT_DIR = "processed_datasettttt"
IMG_SIZE = 256

# Create all necessary output directories
os.makedirs(os.path.join(OUTPUT_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "masks"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "ela_images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "hpf_images"), exist_ok=True)

# ==============================
# 1. Error Level Analysis (ELA)
# ==============================
def generate_ela_image(original_image, quality=90):
    """
    Generates an ELA image using a safe temporary file.
    """
    # Use a temporary file to avoid permission issues in the current directory
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as temp_file:
        temp_path = temp_file.name
        original_image.save(temp_path, 'JPEG', quality=quality)

    try:
        compressed_image = Image.open(temp_path)
        ela_image = ImageChops.difference(original_image, compressed_image)
        
        extrema = ela_image.getextrema()
        max_diff = max([ex[1] for ex in extrema]) if extrema else 0
        scale = 255.0 / max_diff if max_diff != 0 else 1.0
        ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
        
        return np.array(ela_image)
        
    finally:
        # Clean up the temporary file
        if os.path.exists(temp_path):
            os.remove(temp_path)

# ==============================
# 2. High-Frequency Extraction
# ==============================
def high_pass_filter(image_cv):
    """Expects an image in OpenCV (BGR) format."""
    gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return cv2.convertScaleAbs(laplacian)

# ==============================
# 3. Preprocessing Pipeline
# ==============================
def process_single_file(image_path, mask_path, output_idx):
    """
    Loads, processes, and saves a single image and its components.
    Returns True on success, False on failure.
    """
    try:
        # Step 1: Load image using the robust Pillow library
        original_pil_image = Image.open(image_path).convert('RGB')
        
        # Step 2: Convert PIL image to NumPy array, ENFORCING the data type to uint8
        numpy_image_rgb = np.array(original_pil_image, dtype=np.uint8)

        # Step 3: Convert the now-safe NumPy array to OpenCV format (BGR) for processing
        image_cv = cv2.cvtColor(numpy_image_rgb, cv2.COLOR_RGB2BGR)

        # Step 4: Generate ELA from the original PIL image
        ela_img = generate_ela_image(original_pil_image)
        
        # Step 5: Resize all components
        image_resized = cv2.resize(image_cv, (IMG_SIZE, IMG_SIZE))
        ela_resized = cv2.resize(ela_img, (IMG_SIZE, IMG_SIZE))
        high_freq_resized = cv2.resize(high_pass_filter(image_resized), (IMG_SIZE, IMG_SIZE))

        # Step 6: Load and process the mask
        mask_resized_binary = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        if mask_path and os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                resized = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
                mask_resized_binary = (resized > 127).astype(np.uint8) * 255

        # Step 7: Save all processed images
        # Convert resized image from BGR back to RGB for saving with Pillow
        Image.fromarray(cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)).save(os.path.join(OUTPUT_DIR, "images", f"{output_idx}.png"))
        Image.fromarray(ela_resized).save(os.path.join(OUTPUT_DIR, "ela_images", f"{output_idx}.png"))
        Image.fromarray(high_freq_resized).save(os.path.join(OUTPUT_DIR, "hpf_images", f"{output_idx}.png"))
        Image.fromarray(mask_resized_binary).save(os.path.join(OUTPUT_DIR, "masks", f"{output_idx}.png"))
        
        return True # Indicate success
        
    except Exception as e:
        # This broad exception catch handles all possible file processing errors
        tqdm.write(f"Skipping problematic file: {os.path.basename(image_path)} - Reason: {e}")
        return False # Indicate failure

# ==============================
# 4. Processing Dataset
# ==============================
def process_dataset():
    # Corrected folder names based on your original problem description
    tampered_dir = os.path.join(DATA_DIR, "Tp")
    groundtruth_dir = os.path.join(DATA_DIR, "CASIA 2 GroundTruth") 

    # Check if source directories exist before processing
    if not os.path.isdir(tampered_dir):
        print(f"Error: Tampered images directory not found at: {tampered_dir}")
        return
    if not os.path.isdir(groundtruth_dir):
        print(f"Error: Ground truth directory not found at: {groundtruth_dir}")
        return

    image_files = [f for f in os.listdir(tampered_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg', '.tif', '.bmp'))]
    
    saved_count = 0
    for img_name in tqdm(image_files, desc="Processing CASIA2 Dataset"):
        img_path = os.path.join(tampered_dir, img_name)
        base_name, _ = os.path.splitext(img_name)
        mask_name = f"{base_name}_gt.png" 
        mask_path = os.path.join(groundtruth_dir, mask_name)

        # Only increment the saved file index if processing was successful
        if process_single_file(img_path, mask_path, saved_count):
            saved_count += 1

# ==============================
# 5. MAIN EXECUTION
# ==============================
if __name__ == "__main__":
    process_dataset()
    print(f"\nPreprocessing completed! ✅ Files saved to: '{OUTPUT_DIR}'")

Processing CASIA2 Dataset: 100%|███████████████████████████████████████████████████| 5123/5123 [05:31<00:00, 15.44it/s]


Preprocessing completed! ✅ Files saved to: 'processed_datasettttt'
